# Seleção de Amostra para Rotulagem Manual

Este notebook realiza amostragem estratificada de comentários para criação de dataset de treinamento de modelo de classificação.

## Contexto do Processo de Anotação

**Primeira Iteração (600 comentários)**:
- 200 comentários mencionando sintomas
- 200 comentários mencionando termos (nomes de anabolizantes)
- 200 comentários com coocorrência (sintomas + termos)
- **Resultado**: Cohen's Kappa = 0.6 (concordância moderada entre anotadores)

**Segunda Iteração (3000 comentários)** - Este notebook:
- 1000 comentários mencionando sintomas
- 1000 comentários mencionando termos
- 1000 comentários com coocorrência
- Conjuntos disjuntos (sem sobreposição) usando amostragem aleatória
- Comentários da primeira iteração podem estar presentes (devido ao Kappa não ideal)

**Objetivo**: Criar dataset balanceado para treinar modelo que classifica comentários em:
- **Autorrelato**: Relatos pessoais de uso
- **Sintomas/Efeitos Colaterais**: Menções a efeitos adversos

In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv('cleaned_data/final_comments_match_cleaned.csv')

In [3]:
list_terms = ['durateston', 'masteron', 'testo', 'testosterona', 'trembolona', 'oxandrolona', 'oxan', 'primabolan', 'stano', 'hemogenin', 'dianabol', 'trenbolone', 'deca-durabolin', 'boldenona', 'winstrol', 'turinabol', 'proviron', 'halotestin', 'andriol', 'SARMs', 'aplicação de testo', 'ciclo de anabolizante', 'decanoato de nandrolona', 'enantato de testosterona', 'propionato de testosterona', 'cipionato de testosterona', 'undecilenato de boldenona', 'trembo ace']

list_symptoms = {
    "acne": ["espinhas", "erupções cutâneas", "inflamação na pele"],
    "agressividade": ["comportamento agressivo", "explosões de raiva"],
    "alterações no apetite": ["aumento do apetite", "perda de apetite", "distúrbios alimentares"],
    "alterações no fígado": ["lesão hepática", "disfunção hepática", "problemas hepáticos"],
    "alucinações": ["visões", "percepções irreais"],
    "ansiedade": ["preocupação excessiva", "inquietação"],
    "apneia do sono": ["paradas respiratórias durante o sono", "ronco"],
    "atrofia testicular": ["redução do volume testicular", "testículos encolhidos"],
    "aumento da próstata": ["hiperplasia prostática", "próstata aumentada"],
    "aumento de pelos corporais": ["hirsutismo", "crescimento excessivo de pelos"],
    "batimentos irregulares": ["arritmia", "palpitações"],
    "clitoromegalia": ["aumento do clitóris"],
    "colesterol alto": ["hipercolesterolemia"],
    "compulsão": ["uso compulsivo", "desejo incontrolável"],
    "cãibras musculares": ["espasmos musculares"],
    "danos nos rins": ["lesão renal", "nefropatia", "insuficiência renal"],
    "dependência": ["uso contínuo", "não conseguir parar"],
    "depressão": ["tristeza profunda", "humor deprimido", "apatia"],
    "dificuldade para urinar": ["jato fraco", "retenção urinária"],
    "disfunção erétil": ["impotência", "dificuldade de ereção"],
    "distúrbios menstruais": ["menstruação irregular", "amenorreia"],
    "dor de cabeça": ["cefaleia"],
    "dor/inflamação no local da injeção": ["dor no local da aplicação",  "inflamação no local da aplicação", "dor no local da aplicação", "inchaço na aplicação"],
    "dores abdominais": ["dor na barriga", "desconforto abdominal"],
    "dores musculares": ["mialgia"],
    "engrossamento da voz": ["voz mais grave", "voz profunda"],
    "euforia": ["sensação de poder", "excesso de excitação"],
    "fadiga": ["cansaço extremo", "exaustão"],
    "ginecomastia": ["aumento das mamas", "sensibilidade nos mamilos"],
    "hipertrofia": ["crescimento muscular"],
    "hipogonadismo": ["baixa testosterona", "disfunção hormonal masculina"],
    "hipotireoidismo": ["função tireoidiana reduzida"],
    "icterícia": ["pele amarelada", "olhos amarelados"],
    "impotência": ["disfunção erétil", "problemas de ereção"],
    "infarto": ["ataque cardíaco", "infarto do miocárdio"],
    "infertilidade": ["baixa fertilidade", "diminuição da produção de esperma"],
    "insuficiência renal": ["falência renal", "doença renal"],
    "insônia": ["dificuldade para dormir", "sono interrompido"],
    "irritabilidade": ["facilidade para se irritar", "mau humor"],
    "letargia": ["apatia", "lentidão"],
    "mania": ["hiperatividade", "excesso de energia"],
    "náuseas": ["enjoo", "vontade de vomitar"],
    "oleosidade na pele": ["pele oleosa"],
    "oscilações de humor": ["mudanças de humor", "humor instável"],
    "osteoporose": ["ossos frágeis", "redução da densidade óssea"],
    "paranoia": ["desconfiança excessiva", "medo irracional"],
    "pele oleosa": ["brilho excessivo na pele"],
    "pele oleosa excessiva": ["seborréia"],
    "perda de apetite": ["falta de fome", "anorexia"],
    "perda de libido": ["baixa libido", "redução do desejo sexual"],
    "policitemia": ["aumento dos glóbulos vermelhos"],
    "pressão alta": ["hipertensão"],
    "priapismo": ["ereção prolongada"],
    "problemas cardiovasculares": ["doenças do coração", "complicações cardíacas"],
    "psicose": ["quebra da realidade", "delírios"],
    "puberdade precoce": ["maturação sexual antecipada"],
    "queda de cabelo": ["alopecia", "calvície"],
    "retenção de líquido": ["inchaço", "edema"],
    "suor excessivo": ["hiperidrose", "suor noturno"],
    "toxicidade hepática": ["dano hepático", "hepatite medicamentosa"],
    "tremores": ["movimentos involuntários"],
    "trombose": ["coágulos sanguíneos"],
    "tumores hepáticos": ["neoplasias hepáticas", "câncer no fígado"],
    "voz grossa": ["voz masculina", "mudança vocal"],
    "vômitos": ["emese"]
}

---
## 1. Definição de Termos e Sintomas

Listas curadas de:
- **list_terms**: Nomes de anabolizantes e compostos relacionados (durateston, oxandrolona, SARMs, etc.)
- **list_symptoms**: Dicionário de sintomas/efeitos colaterais com variações linguísticas (ex: "ginecomastia" inclui "aumento das mamas", "sensibilidade nos mamilos")

Estas listas serão usadas para identificar comentários relevantes via busca por padrões regex.

In [4]:
all_symptoms = []
for main, variations in list_symptoms.items():
    all_symptoms.append(main)
    all_symptoms.extend(variations)

pattern_symptoms = '|'.join([r'\b' + re.escape(symptom) + r'\b' for symptom in all_symptoms])
mask_symptoms = df['comment'].str.contains(pattern_symptoms, case=False, na=False)

pattern_terms = '|'.join([r'\b' + re.escape(term) + r'\b' for term in list_terms])
mask_termos = df['comment'].str.contains(pattern_terms, case=False, na=False)

---
## 2. Filtragem de Comentários

Cria máscaras booleanas para identificar comentários que mencionam:
- **mask_symptoms**: Qualquer sintoma da lista (termos principais + variações)
- **mask_termos**: Qualquer anabolizante da lista
- **Coocorrência**: Comentários que mencionam ambos (sintomas AND termos)

Usa regex com `\b` (word boundaries) para evitar correspondências parciais.

In [5]:
df = df[['comment_id', 'comment']]

df_symptoms = df[mask_symptoms]
df_termos = df[mask_termos]
df_coocurrence = df[mask_symptoms & mask_termos]

In [6]:
print(f'Total de comentários mencionando termos: {df_termos.shape[0]}')
print(f'Total de comentários mencionando sintomas: {df_symptoms.shape[0]}')
print(f'Total de comentários com coocorrência: {df_coocurrence.shape[0]}')

Total de comentários mencionando termos: 90901
Total de comentários mencionando sintomas: 17976
Total de comentários com coocorrência: 6813


In [7]:
amostra_cooc = df_coocurrence.sample(n=1000, random_state=42)

# Remove IDs sorteados da coocorrência dos outros dois
df_symptoms = df_symptoms[~df_symptoms['comment_id'].isin(amostra_cooc['comment_id'])]
df_termos = df_termos[~df_termos['comment_id'].isin(amostra_cooc['comment_id'])]

amostra_symptoms = df_symptoms.sample(n=1000, random_state=42)

# Remove IDs sorteados de sintomas dos termos
df_termos = df_termos[~df_termos['comment_id'].isin(amostra_symptoms['comment_id'])]

amostra_termos = df_termos.sample(n=1000, random_state=42)

---
## 3. Amostragem Estratificada Disjunta

Processo sequencial para garantir conjuntos sem sobreposição:
1. **Amostra coocorrência** (1000): Sorteia aleatoriamente da interseção
2. **Remove IDs** da coocorrência dos outros dois conjuntos
3. **Amostra sintomas** (1000): Sorteia dos comentários restantes
4. **Remove IDs** de sintomas do conjunto de termos
5. **Amostra termos** (1000): Sorteia dos comentários restantes

**Resultado**: 3000 comentários únicos e disjuntos (random_state=42 para reprodutibilidade).

In [8]:
amostra_cooc = amostra_cooc.reset_index(drop=True)
amostra_symptoms = amostra_symptoms.reset_index(drop=True)
amostra_termos = amostra_termos.reset_index(drop=True)

In [9]:
df_annotations = pd.concat([amostra_cooc, amostra_symptoms, amostra_termos], ignore_index=True)

In [10]:
df_annotations = df_annotations.sample(frac=1, random_state=42).reset_index(drop=True)

---
## 4. Preparação Final

- **Concatenação**: Combina as 3 amostras em um único dataset
- **Shuffle**: Embaralha os comentários para evitar viés de ordem durante anotação
- **Validação**: Verifica ausência de duplicatas
- **Output**: Dataset pronto para distribuição entre anotadores (`data_to_label.csv`)

In [11]:
df_annotations.head()

,comment_id,comment
0,Ugx7QBrAqlHD-gNwWud4AaABAg,Eu tenho 41 anos tom com disfunção erétil perd...
1,UgwVhgrgq7cyiXmx8rZ4AaABAg.9LivuUpAgEC9Ov33-Wlkwz,"@@marcilenemedeiros5000 Olá, tive muita fome, ..."
2,UgwWbphzRqPckmAhggt4AaABAg.AA6pjoqTFbPAATRxlrgVin,"​@@MatheusAvelar7é com o tempo, vai acontecer ..."
3,UgzYBcGSAu0DfKWAVe14AaABAg.A3CdTDOx4WxA3DPD-jki4j,Pelo visto você não toma hormônio né mano. Apl...
4,UgyuL5D1bra_n6P2psF4AaABAg.9hm0O4K-7Wq9hu6y1B-zB8,@@3DPlayGames cara eu tenho mt medo de usar tr...


In [13]:
df_annotations.shape

(3000, 2)

In [14]:
df_annotations['comment_id'].duplicated().sum()

0

In [12]:
# df_annotations.to_csv('annotation/data_to_label.csv', index=False)